# Stress Prediction v21 — Triple-Model Ensemble (LGBM + CatBoost + XGBoost)

Friend's LB 0.50 → ceiling is achievable. We're under-extracting. Strategy: KEEP v16's proven structure, ADD model diversity through peer-strength gradient boosters (CatBoost, XGBoost), trained identically.

## What's preserved from v16 (proven 0.377)
- v7c feature extractor (3-min window, raw stats + accel mag + HRV)
- `pid_enc` feature kept
- StratifiedKFold 5-fold × 7 seeds = 35 base models per algorithm
- `class_weight='balanced'` + `sample_weight`
- Calibration `proba × train_prior^alpha`, default α=1.6
- Session smoothing strength=0.30

## What's new
- **CatBoost** trained identically (35 more models, ordered boosting, different inductive bias)
- **XGBoost** trained identically (35 more models, different gradient impl)
- Total: **105 base models** ensembled by soft-probability average
- Multiple alpha calibration variants saved for selection (1.4, 1.6, 1.8, 2.0)

## Why this should work where v19 didn't
v19 added HistGradientBoosting + ExtraTrees (weaker than LGBM); ensemble weights collapsed to LGBM=1.0. CatBoost and XGBoost are **peer-strength** to LightGBM with genuinely different decision boundaries (CatBoost: ordered boosting; XGBoost: different split logic + L2). They won't be ignored by the ensemble.

Expected runtime: ~30-45 minutes.


In [1]:
%pip -q install lightgbm catboost xgboost scikit-learn pandas numpy scipy


[notice] A new release of pip is available: 26.0 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from scipy import stats as spstats

from sklearn.impute import SimpleImputer
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold

import lightgbm as lgb
import catboost as cb
import xgboost as xgb

warnings.filterwarnings('ignore')
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

DATA_DIR = Path('.')
TRAIN_DATA  = pd.read_csv(DATA_DIR / 'train-sensor.csv')
TRAIN_LABEL = pd.read_csv(DATA_DIR / 'train-label.csv')
TEST_DATA   = pd.read_csv(DATA_DIR / 'test-sensor.csv')
TEST_LABEL  = pd.read_csv(DATA_DIR / 'test-label.csv')

print('Raw shapes')
print('  TRAIN_DATA :', TRAIN_DATA.shape)
print('  TRAIN_LABEL:', TRAIN_LABEL.shape)
print('  TEST_DATA  :', TEST_DATA.shape)
print('  TEST_LABEL :', TEST_LABEL.shape)


Raw shapes
  TRAIN_DATA : (4694400, 8)
  TRAIN_LABEL: (815, 4)
  TEST_DATA  : (5921280, 8)
  TEST_LABEL : (1028, 4)


In [3]:
SENSOR_COLS = ['accel_x', 'accel_y', 'accel_z', 'eda', 'heart_rate', 'temperature']

def clean_sensor(df):
    out = df.copy()
    out['pid'] = out['pid'].astype(str)
    out['timestamp'] = pd.to_numeric(out['timestamp'], errors='coerce').astype(float)
    for c in SENSOR_COLS:
        out[c] = pd.to_numeric(out[c], errors='coerce').astype(float)
    out['accel_x'] = out['accel_x'].clip(-128, 127)
    out['accel_y'] = out['accel_y'].clip(-128, 127)
    out['accel_z'] = out['accel_z'].clip(-128, 127)
    out['eda'] = out['eda'].clip(0, 60)
    out['heart_rate'] = out['heart_rate'].clip(40, 190)
    out['temperature'] = out['temperature'].clip(20, 40)
    return out.sort_values(['pid', 'timestamp']).reset_index(drop=True)

def clean_label(df):
    out = df.copy()
    out['id'] = pd.to_numeric(out['id'], errors='raise').astype(int)
    out['pid'] = out['pid'].astype(str)
    out['timestamp'] = pd.to_numeric(out['timestamp'], errors='coerce').astype(float)
    out['stress'] = pd.to_numeric(out['stress'], errors='coerce')
    return out

TRAIN_DATA = clean_sensor(TRAIN_DATA)
TEST_DATA  = clean_sensor(TEST_DATA)
TRAIN_LABEL = clean_label(TRAIN_LABEL)
TEST_LABEL  = clean_label(TEST_LABEL)
print('Cleaned.')


Cleaned.


## Feature Extraction (v16 proven features)

In [4]:
WINDOW_MS = 180_000
HALF_MS   = 90_000
THIRD_MS  = 60_000

def hrv_time_domain(bpm_series):
    f = {}
    bpm = bpm_series.dropna().values.astype(float)
    if len(bpm) < 10:
        for k in ['sdnn','rmssd','pnn25','pnn50','mean_rr','cv_rr']:
            f['hrv_' + k] = np.nan
        return f
    bpm_1hz = bpm[::32] if len(bpm) >= 32 else bpm
    rr = 60000.0 / np.clip(bpm_1hz, 30, 220)
    rr_diff = np.diff(rr)
    f['hrv_sdnn'] = float(np.std(rr))
    f['hrv_rmssd'] = float(np.sqrt(np.mean(rr_diff ** 2))) if len(rr_diff) else 0.0
    f['hrv_pnn25'] = float(np.mean(np.abs(rr_diff) > 25)) * 100 if len(rr_diff) else 0.0
    f['hrv_pnn50'] = float(np.mean(np.abs(rr_diff) > 50)) * 100 if len(rr_diff) else 0.0
    f['hrv_mean_rr'] = float(np.mean(rr))
    f['hrv_cv_rr'] = f['hrv_sdnn'] / f['hrv_mean_rr'] if f['hrv_mean_rr'] > 1e-6 else 0.0
    return f

def extract_features(label_df, sensor_df, pid_enc_map):
    sensor_by_pid = {pid: grp.sort_values('timestamp').reset_index(drop=True)
                     for pid, grp in sensor_df.groupby('pid')}
    rows = []
    for n, lrow in enumerate(label_df.itertuples(index=False), 1):
        pid = lrow.pid; ts = float(lrow.timestamp); lid = int(lrow.id)
        feat = {'id': lid}
        sg = sensor_by_pid.get(pid)
        if sg is None:
            rows.append(feat); continue
        ta = sg['timestamp'].values
        wa  = sg.loc[(ta >= ts - WINDOW_MS) & (ta <= ts), SENSOR_COLS]
        wf  = sg.loc[(ta >= ts - WINDOW_MS) & (ta < ts - HALF_MS), SENSOR_COLS]
        wl  = sg.loc[(ta >= ts - HALF_MS) & (ta <= ts), SENSOR_COLS]
        wt1 = sg.loc[(ta >= ts - WINDOW_MS) & (ta < ts - 2*THIRD_MS), SENSOR_COLS]
        wt3 = sg.loc[(ta >= ts - THIRD_MS) & (ta <= ts), SENSOR_COLS]
        feat['window_count'] = len(wa)
        for c in SENSOR_COLS:
            v = wa[c].dropna().values.astype(float)
            vf = wf[c].dropna().values.astype(float)
            vl = wl[c].dropna().values.astype(float)
            vt1 = wt1[c].dropna().values.astype(float)
            vt3 = wt3[c].dropna().values.astype(float)
            if len(v) == 0:
                for s in ['mean','std','min','max','median','skew','kurt','range','q25','q75','iqr','delta','slope','t1_mean','t3_mean','t3t1']:
                    feat[f'{c}_{s}'] = np.nan
                continue
            feat[f'{c}_mean'] = float(np.mean(v))
            feat[f'{c}_std'] = float(np.std(v))
            feat[f'{c}_min'] = float(np.min(v))
            feat[f'{c}_max'] = float(np.max(v))
            feat[f'{c}_median'] = float(np.median(v))
            feat[f'{c}_skew'] = float(spstats.skew(v)) if len(v) > 2 else 0.0
            feat[f'{c}_kurt'] = float(spstats.kurtosis(v)) if len(v) > 2 else 0.0
            feat[f'{c}_range'] = float(np.max(v) - np.min(v))
            feat[f'{c}_q25'] = float(np.percentile(v, 25))
            feat[f'{c}_q75'] = float(np.percentile(v, 75))
            feat[f'{c}_iqr'] = feat[f'{c}_q75'] - feat[f'{c}_q25']
            feat[f'{c}_delta'] = float(np.mean(vl) - np.mean(vf)) if len(vf) and len(vl) else 0.0
            feat[f'{c}_slope'] = float(np.polyfit(np.linspace(0, 1, len(v)), v, 1)[0]) if len(v) > 2 else 0.0
            feat[f'{c}_t1_mean'] = float(np.mean(vt1)) if len(vt1) else float(np.mean(v))
            feat[f'{c}_t3_mean'] = float(np.mean(vt3)) if len(vt3) else float(np.mean(v))
            feat[f'{c}_t3t1'] = feat[f'{c}_t3_mean'] - feat[f'{c}_t1_mean']
        ax, ay, az = wa['accel_x'].values, wa['accel_y'].values, wa['accel_z'].values
        if len(ax):
            mag = np.sqrt(ax**2 + ay**2 + az**2)
            feat['accel_mag_mean'] = float(np.mean(mag))
            feat['accel_mag_std']  = float(np.std(mag))
            feat['accel_mag_max']  = float(np.max(mag))
        else:
            feat['accel_mag_mean'] = feat['accel_mag_std'] = feat['accel_mag_max'] = np.nan
        feat.update(hrv_time_domain(wa['heart_rate']))
        feat['pid_enc'] = pid_enc_map.get(pid, -1)
        rows.append(feat)
        if n % 200 == 0:
            print(f'  {n}/{len(label_df)}')
    return pd.DataFrame(rows).set_index('id')

train_pid_map = {p: i for i, p in enumerate(TRAIN_LABEL['pid'].unique())}
print('Extracting train features...')
train_features = extract_features(TRAIN_LABEL, TRAIN_DATA, train_pid_map)
print('Extracting test features...')
test_features  = extract_features(TEST_LABEL,  TEST_DATA,  train_pid_map)
print('train:', train_features.shape, 'test:', test_features.shape)


Extracting train features...
  200/815
  400/815
  600/815
  800/815
Extracting test features...
  200/1028
  400/1028
  600/1028
  800/1028
  1000/1028
train: (815, 107) test: (1028, 107)


In [5]:
tli = TRAIN_LABEL.set_index('id')
y = tli.loc[train_features.index, 'stress'].astype(int)

imputer = SimpleImputer(strategy='median')
X_imp      = pd.DataFrame(imputer.fit_transform(train_features), columns=train_features.columns, index=train_features.index)
X_test_imp = pd.DataFrame(imputer.transform(test_features),       columns=test_features.columns,  index=test_features.index)

counts = Counter(y); total = len(y)
class_weights = {0: total / (3*counts[0]),
                 1: min(total / (3*counts[1]), 2.5),
                 2: total / (3*counts[2])}
sample_weights = np.array([class_weights[int(yi)] for yi in y])
train_prior = np.array([counts[i]/total for i in range(3)])

print('X_imp:', X_imp.shape, '| Class weights:', {k:round(v,3) for k,v in class_weights.items()})
print('Train prior:', train_prior.round(3).tolist())


X_imp: (815, 107) | Class weights: {0: 1.677, 1: 2.5, 2: 0.463}
Train prior: [0.199, 0.081, 0.72]


## Train 3 Model Families × 7 Seeds × 5 Folds = 105 Models

Same StratifiedKFold structure for all three to ensure fair averaging.

In [6]:
# Session helpers (for final smoothing only)
def make_session_groups(label_df, gap_ms=30 * 60 * 1000):
    labels = label_df.copy().reset_index(drop=True)
    labels['rowpos'] = np.arange(len(labels))
    out = []
    for pid, grp in labels.sort_values(['pid','timestamp']).groupby('pid', sort=False):
        ts = grp['timestamp'].values.astype(float)
        sess = np.cumsum(np.r_[0, np.diff(ts) > gap_ms])
        for sid in np.unique(sess):
            out.append(grp['rowpos'].values[sess == sid])
    return out

def smooth_by_session(proba, sessions, strength=0.30):
    out = proba.copy()
    for idx in sessions:
        mean = proba[idx].mean(axis=0, keepdims=True)
        out[idx] = (1 - strength) * proba[idx] + strength * mean
    return out

train_label_for_rows = TRAIN_LABEL.set_index('id').loc[X_imp.index].reset_index()
TRAIN_SESSIONS = make_session_groups(train_label_for_rows)
TEST_SESSIONS  = make_session_groups(TEST_LABEL)
print('Train sessions:', len(TRAIN_SESSIONS), 'Test sessions:', len(TEST_SESSIONS))

# === LightGBM (v16 proven params) ===
LGBM_PARAMS = dict(
    n_estimators=1000, learning_rate=0.02, num_leaves=127, max_depth=-1,
    min_child_samples=5, subsample=0.6, colsample_bytree=0.6,
    reg_alpha=0.3, reg_lambda=0.3,
    class_weight='balanced', objective='multiclass', num_class=3,
    n_jobs=-1, verbose=-1,
)

# === CatBoost ===
CB_PARAMS = dict(
    iterations=1000, learning_rate=0.03, depth=6,
    l2_leaf_reg=3.0, bagging_temperature=0.5,
    auto_class_weights='Balanced',
    loss_function='MultiClass', eval_metric='MultiClass',
    od_type='Iter', od_wait=50,
    verbose=False, allow_writing_files=False,
)

# === XGBoost ===
XGB_PARAMS = dict(
    n_estimators=1000, learning_rate=0.03, max_depth=6,
    min_child_weight=3, subsample=0.7, colsample_bytree=0.7,
    reg_alpha=0.3, reg_lambda=1.0,
    objective='multi:softprob', num_class=3,
    eval_metric='mlogloss', tree_method='hist',
    n_jobs=-1, verbosity=0,
)

SEEDS = [42, 7, 123, 17, 99, 256, 314]
N_SPLITS = 5
n_test = len(X_test_imp)

test_proba_lgbm = np.zeros((n_test, 3))
test_proba_cb   = np.zeros((n_test, 3))
test_proba_xgb  = np.zeros((n_test, 3))

cv_scores_lgbm = []; cv_scores_cb = []; cv_scores_xgb = []

for seed in SEEDS:
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    seed_lgbm = np.zeros((n_test, 3)); seed_cb = np.zeros((n_test, 3)); seed_xgb = np.zeros((n_test, 3))
    fold_lgbm = []; fold_cb = []; fold_xgb = []
    
    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_imp, y), 1):
        X_tr, y_tr = X_imp.iloc[tr_idx], y.iloc[tr_idx]
        X_va, y_va = X_imp.iloc[va_idx], y.iloc[va_idx]
        sw_tr = sample_weights[tr_idx]
        
        # LightGBM
        m_lgbm = lgb.LGBMClassifier(**{**LGBM_PARAMS, 'random_state': seed})
        m_lgbm.fit(X_tr, y_tr, sample_weight=sw_tr,
                   eval_set=[(X_va, y_va)],
                   callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(-1)])
        seed_lgbm += m_lgbm.predict_proba(X_test_imp)
        fold_lgbm.append(balanced_accuracy_score(y_va, m_lgbm.predict(X_va)))
        
        # CatBoost
        m_cb = cb.CatBoostClassifier(**{**CB_PARAMS, 'random_seed': seed})
        m_cb.fit(X_tr, y_tr, sample_weight=sw_tr,
                 eval_set=(X_va, y_va), use_best_model=True)
        seed_cb += m_cb.predict_proba(X_test_imp)
        fold_cb.append(balanced_accuracy_score(y_va, m_cb.predict(X_va).astype(int).flatten()))
        
        # XGBoost
        m_xgb = xgb.XGBClassifier(**{**XGB_PARAMS, 'random_state': seed,
                                     'early_stopping_rounds': 100})
        m_xgb.fit(X_tr, y_tr, sample_weight=sw_tr,
                  eval_set=[(X_va, y_va)], verbose=False)
        seed_xgb += m_xgb.predict_proba(X_test_imp)
        fold_xgb.append(balanced_accuracy_score(y_va, m_xgb.predict(X_va)))
    
    seed_lgbm /= N_SPLITS; seed_cb /= N_SPLITS; seed_xgb /= N_SPLITS
    test_proba_lgbm += seed_lgbm; test_proba_cb += seed_cb; test_proba_xgb += seed_xgb
    cv_scores_lgbm.append(np.mean(fold_lgbm))
    cv_scores_cb.append(np.mean(fold_cb))
    cv_scores_xgb.append(np.mean(fold_xgb))
    print(f'Seed {seed}: leaky CV   LGBM={np.mean(fold_lgbm):.4f}  CB={np.mean(fold_cb):.4f}  XGB={np.mean(fold_xgb):.4f}')

test_proba_lgbm /= len(SEEDS); test_proba_cb /= len(SEEDS); test_proba_xgb /= len(SEEDS)

print(f'\nMean leaky CV: LGBM={np.mean(cv_scores_lgbm):.4f}  CB={np.mean(cv_scores_cb):.4f}  XGB={np.mean(cv_scores_xgb):.4f}')
print(f'Raw test argmax dist by model:')
print(f'  LGBM    : {dict(Counter(test_proba_lgbm.argmax(1)))}')
print(f'  CatBoost: {dict(Counter(test_proba_cb.argmax(1)))}')
print(f'  XGBoost : {dict(Counter(test_proba_xgb.argmax(1)))}')

# Ensemble: equal-weighted soft average of the three
test_proba_ens = (test_proba_lgbm + test_proba_cb + test_proba_xgb) / 3.0
print(f'  Ensemble: {dict(Counter(test_proba_ens.argmax(1)))}')


Train sessions: 67 Test sessions: 106
Seed 42: leaky CV   LGBM=0.8121  CB=0.8087  XGB=0.7928
Seed 7: leaky CV   LGBM=0.8126  CB=0.8003  XGB=0.7833
Seed 123: leaky CV   LGBM=0.7910  CB=0.8055  XGB=0.7904
Seed 17: leaky CV   LGBM=0.8044  CB=0.7997  XGB=0.7925
Seed 99: leaky CV   LGBM=0.8079  CB=0.8078  XGB=0.7972
Seed 256: leaky CV   LGBM=0.8062  CB=0.8087  XGB=0.7828
Seed 314: leaky CV   LGBM=0.7949  CB=0.8136  XGB=0.7745

Mean leaky CV: LGBM=0.8042  CB=0.8063  XGB=0.7876
Raw test argmax dist by model:
  LGBM    : {np.int64(2): 137, np.int64(0): 371, np.int64(1): 520}
  CatBoost: {np.int64(2): 231, np.int64(0): 516, np.int64(1): 281}
  XGBoost : {np.int64(2): 260, np.int64(0): 432, np.int64(1): 336}
  Ensemble: {np.int64(2): 199, np.int64(0): 440, np.int64(1): 389}


## Calibration Sweep — Save Multiple Submissions

We save submissions at multiple alpha values so you can pick based on which test distribution best matches train prior.

In [7]:
def make_submission(proba, alpha, smooth_strength, sessions, prior, fname):
    cal = proba * (prior ** alpha)
    cal = cal / cal.sum(axis=1, keepdims=True)
    if smooth_strength > 0:
        cal = smooth_by_session(cal, sessions, strength=smooth_strength)
    preds = np.argmax(cal, axis=1).astype(int)
    pd.DataFrame({'id': TEST_LABEL['id'].values, 'stress': preds}).to_csv(fname, index=False)
    counts = np.bincount(preds, minlength=3)
    fracs = counts / len(preds)
    dev = np.abs(fracs - prior).max()
    return preds, counts, fracs, dev

print(f'{"submission":>40s}  {"alpha":>5s} {"smooth":>6s}  {"dist (0/1/2)":>22s}  {"dev":>5s}')

# Main candidate: ensemble + alpha=1.6 + smooth=0.30 (v16 proven calibration on triple ensemble)
for alpha in [1.4, 1.6, 1.8, 2.0]:
    fname = f'submission_ens_alpha_{alpha}.csv'
    preds, counts, fracs, dev = make_submission(test_proba_ens, alpha, 0.30, TEST_SESSIONS, train_prior, fname)
    print(f'{fname:>40s}  {alpha:>5.2f} {0.30:>6.2f}  {str(counts.tolist()):>22s}  {dev:>5.3f}')

# Individual model submissions (in case ensemble underperforms a single model)
for name, proba in [('lgbm', test_proba_lgbm), ('cb', test_proba_cb), ('xgb', test_proba_xgb)]:
    fname = f'submission_{name}_alpha_1.6.csv'
    preds, counts, fracs, dev = make_submission(proba, 1.6, 0.30, TEST_SESSIONS, train_prior, fname)
    print(f'{fname:>40s}  {1.6:>5.2f} {0.30:>6.2f}  {str(counts.tolist()):>22s}  {dev:>5.3f}')

# Default submission.csv = ensemble alpha=1.6 smooth=0.30 (the proven calibration applied to triple ensemble)
preds, counts, fracs, dev = make_submission(test_proba_ens, 1.6, 0.30, TEST_SESSIONS, train_prior, 'submission.csv')
print(f'\n>>> DEFAULT submission.csv: ensemble + alpha=1.6 + smooth=0.30')
print(f'    Distribution: {counts.tolist()}, fractions {fracs.round(3).tolist()}, dev {dev:.3f}')
print(f'    Train prior : {train_prior.round(3).tolist()}')

print('\nRECOMMENDED: pick the alpha whose distribution is closest to train prior (lowest dev).')
print('If multiple alphas have low dev, prefer the one closest to alpha=1.6 (proven anchor).')


                              submission  alpha smooth            dist (0/1/2)    dev
            submission_ens_alpha_1.4.csv   1.40   0.30           [57, 10, 961]  0.215
            submission_ens_alpha_1.6.csv   1.60   0.30            [28, 2, 998]  0.251
            submission_ens_alpha_1.8.csv   1.80   0.30           [11, 1, 1016]  0.268
            submission_ens_alpha_2.0.csv   2.00   0.30            [8, 0, 1020]  0.272
           submission_lgbm_alpha_1.6.csv   1.60   0.30          [160, 41, 827]  0.084
             submission_cb_alpha_1.6.csv   1.60   0.30            [9, 0, 1019]  0.271
            submission_xgb_alpha_1.6.csv   1.60   0.30            [34, 3, 991]  0.244

>>> DEFAULT submission.csv: ensemble + alpha=1.6 + smooth=0.30
    Distribution: [28, 2, 998], fractions [0.027, 0.002, 0.971], dev 0.251
    Train prior : [0.199, 0.081, 0.72]

RECOMMENDED: pick the alpha whose distribution is closest to train prior (lowest dev).
If multiple alphas have low dev, prefer the on

## Summary

In [8]:
print('========== v21 SUMMARY ==========')
print(f'Mean leaky CV BAs:')
print(f'  LightGBM  : {np.mean(cv_scores_lgbm):.4f}')
print(f'  CatBoost  : {np.mean(cv_scores_cb):.4f}')
print(f'  XGBoost   : {np.mean(cv_scores_xgb):.4f}')
print()
print('Saved candidate submissions:')
print('  submission.csv                  (default: ensemble + alpha=1.6 + smooth=0.30)')
print('  submission_ens_alpha_1.4.csv    (lighter calibration)')
print('  submission_ens_alpha_1.6.csv    (= submission.csv, v16 proven anchor)')
print('  submission_ens_alpha_1.8.csv    (heavier calibration)')
print('  submission_ens_alpha_2.0.csv    (heaviest calibration)')
print('  submission_lgbm_alpha_1.6.csv   (LGBM only, fallback)')
print('  submission_cb_alpha_1.6.csv     (CatBoost only, fallback)')
print('  submission_xgb_alpha_1.6.csv    (XGBoost only, fallback)')
print()
print('PICK STRATEGY:')
print('  1. Compare all distributions vs train prior [0.199, 0.081, 0.720].')
print('  2. Submit the file with lowest dev that has dist closest to (~17/8/75).')
print('  3. Default submission.csv (alpha=1.6) is the proven anchor; only pick another')
print('     if its distribution is meaningfully closer to prior.')
print('==================================')


========== v21 SUMMARY ==========
Mean leaky CV BAs:
  LightGBM  : 0.8042
  CatBoost  : 0.8063
  XGBoost   : 0.7876

Saved candidate submissions:
  submission.csv                  (default: ensemble + alpha=1.6 + smooth=0.30)
  submission_ens_alpha_1.4.csv    (lighter calibration)
  submission_ens_alpha_1.6.csv    (= submission.csv, v16 proven anchor)
  submission_ens_alpha_1.8.csv    (heavier calibration)
  submission_ens_alpha_2.0.csv    (heaviest calibration)
  submission_lgbm_alpha_1.6.csv   (LGBM only, fallback)
  submission_cb_alpha_1.6.csv     (CatBoost only, fallback)
  submission_xgb_alpha_1.6.csv    (XGBoost only, fallback)

PICK STRATEGY:
  1. Compare all distributions vs train prior [0.199, 0.081, 0.720].
  2. Submit the file with lowest dev that has dist closest to (~17/8/75).
  3. Default submission.csv (alpha=1.6) is the proven anchor; only pick another
     if its distribution is meaningfully closer to prior.
